## Congo Basin shapefile recycling
- running with merged data (zero evap over ocean) and Congo Basin shapefile
- running with rotation options and plotting seasonal averages for all rotations

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import scipy
import sys 
import warnings
warnings.filterwarnings('ignore')
import geopandas as gpd
from shapely.geometry import mapping
import cartopy.crs as ccrs
import cartopy.feature
import matplotlib.pyplot as plt
import time as timer
start_all = timer.time()

### Read in surface and pressure level files
**Surface file options:**
- S_SE (surface vars including only surface evaporation)
- S_LSE (surface vars including land and surface evaporation)

**Pressure level file options:**
- L_M (pressure level vars resampled to monthly timestep)
- L_HI (pressure level vars with integrated moist flux calcualted hourly and then resampled to monthly timestep)

In [ ]:
YR = 1990

dataf ="/Volumes/blue_wd/ESA_F4R/2026_mergeds/" 
datao ="/Volumes/blue_wd/ESA_F4R/2026_rho/cb/" 
datap ="/Volumes/blue_wd/ESA_F4R/2026_plots/cb/" 
datas ="/Users/ellendyer/Library/Mobile Documents/com~apple~CloudDocs/1SHARED_WORK/Work/3_ESA_GRANT/MODEL/Shapefiles/"
shp_cod = gpd.read_file(datas+"congo_basin_evergreen.shp")

S_NAME = "S_SE" # S_SE or S_LSE 
L_NAME = "L_M" # L_M or L_HI

In [ ]:
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.RIVERS)
ax.add_feature(cartopy.feature.OCEAN)
shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
plt.show()
plt.clf()

#### **Read in pre-processed files that have the following conversions:**

**Read in ERA5 data on pressure levels**
- resampled to monthly MS timestep
- shum multiplied by 1000 to convert from kg/kg --> g/kg
- pressure levels are divided by 100 to convert from Pa to hPa (only for fortnightly files)
- sort data by descending pressure levels (only for fortnightly files)


**Input file units:**
- plev - pa
- q - kg/kg
- u - m/s
- v - m/s

** ***Might include Fx and Fy if L_HI is being read in***

**Read in ERA5 land and surface data**
- Land: selecting hour 23 (0-23) of Prec and Evap because of how ERA5 Land variables are accumulated (https://confluence.ecmwf.int/pages/viewpage.action?pageId=197702790 - https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A+data+documentation#ERA5Land:datadocumentation-accumulationsAccumulations)
- Surface: evap is accumulated per hour so a daily sum is calculated
- prec is multiplied by 1000 to convert from m to mm
- evap is multiplied by -1000 to convert from m to mm and upward fluxes in land model are considered negative
- Prec, Evap, and Psfc are then resampled to MS monthly and also interpolated to coarser pressure level grid

**Input file units:**
- tp - m (no longer reading in)
- e - m (-)
- sp - pa

**Merging all input datasets into one dataset for recyling code called ds**
- close both input datasets
- sort everything so latitude is south to north
- transpose dimensions so they run (lon,lat,level,time) as in recycling code
- save input ds to file

**Integrate zonal and meridional moisture flux**
- **Must check if there are any nans in input arrays** - there can be none because we are using nans as an indicator in the modified definitions

In [ ]:
dsS = xr.open_dataset(dataf+"merge_erads_"+S_NAME+"_"+str(YR)+".nc")
dsL = xr.open_dataset(dataf+"merge_erads_"+L_NAME+"_"+str(YR)+".nc")
ds = xr.merge([dsS,dsL])

#Running the code with an irregular boundary means that there will be nans once the data is clipped
#For this to work properly it is important that there are no nans in the input data which would 
#actually be missing data where recycling will be calculated
print(np.isnan(ds['Evap_all'].values).any())
print(ds['Evap_all'].isnull().count().values)
print('Number of nans in Evap: ', ds['Evap_all'].where(np.isnan(ds['Evap_all'])==True,drop=True).count().values)
print('Number of nans in Psfc: ', ds['Psfc'].where(np.isnan(ds['Psfc'])==True,drop=True).count().values)
print('Number of nans in Uwnd: ', ds['Uwnd'].where(np.isnan(ds['Uwnd'])==True,drop=True).count().values)
print('Number of nans in Vwnd: ', ds['Vwnd'].where(np.isnan(ds['Vwnd'])==True,drop=True).count().values)
print('Number of nans in Shum: ', ds['Shum'].where(np.isnan(ds['Shum'])==True,drop=True).count().values)

#Arrange data and prepare it to be clipped by the shapefile
ds = ds.transpose("time","level","lat","lon",missing_dims='ignore')
#ds = ds.rio.set_spatial_dims(x_dim="lon",y_dim="lat")
ds = ds.rename({'lon': 'x','lat': 'y'})
ds.rio.write_crs("epsg:4326", inplace=True)
ds['Evap_cb'] = ds['Evap_all'].rio.clip(shp_cod.geometry.apply(mapping),shp_cod.crs,drop=False)
ds = ds.rename({'x': 'lon','y': 'lat'})
ds['Evap_cb'] = ds['Evap_cb'].fillna(0.0)

ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.OCEAN)
shp_cod.plot(ax=ax, edgecolor='yellow',facecolor='none',lw=2,zorder=2,linestyle='--')
ds['Evap_cb'].mean('time').plot(ax=ax,transform=ccrs.PlateCarree(),
                                       add_colorbar=True,
                                       #vmin=0,vmax=2,
                                       alpha=1,
                                       cmap=plt.cm.gist_earth_r,
                                       extend="both")

ax.set_extent([8, 31, -8, 8])
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()
          
#Points that are 0.0 are inside the region, points on the boundary will be
ds = ds.transpose("lon","lat","level","time",missing_dims='ignore')

ds = ds.sel(lon=slice(7,31),lat=slice(-8,8))

print(np.isnan(ds['Evap_cb'].values).any())
print(ds['Evap_cb'].isnull().count().values)

In [ ]:
if L_NAME=='L_HI':
    Fx = ds['Fx']
    Fy = ds['Fy']
else:
    #Prepping datasets near surface for recycling
    import bulk_recycling_model.numerical_integration
    
    # Integrate 10^-3 Shum Uwnd dp
    # Because the integration limits are from high pressure to low pressure, we need to invert the sign.
    integrand = -1 * 1e-3 * ds["Shum"] * ds["Uwnd"]
    Fx = bulk_recycling_model.numerical_integration.integrate_with_extrapolation(integrand, ds["Psfc"])
    # Units: mb x m/s
    
    # Integrate 10^-3 Shum Vwnd dp
    # Because the integration limits are from high pressure to low pressure, we need to invert the sign.
    integrand = -1 * 1e-3 * ds["Shum"] * ds["Vwnd"]
    Fy = bulk_recycling_model.numerical_integration.integrate_with_extrapolation(integrand, ds["Psfc"])
    # Units: mb x m/s


**Prepare scaled data for recycling code**
- Evaporation and moisture fluxes


In [ ]:
# Prepare and scale the data
from bulk_recycling_model import preprocess
from bulk_recycling_model.axis import Axis
from bulk_recycling_model.scaling import Scaling, UnitSystem

# degrees
L = ds.coords["lon"].max().item() - ds.coords["lon"].min().item()
# convert to meters
L = L * 111e3 * np.cos(np.deg2rad(ds.coords["lat"].mean().item()))
dx = L / ds.sizes["lon"]

# lon axis
lon_axis = Axis(
    ds.coords["lon"].min().item(),
    ds.coords["lon"].diff("lon").mean().item(),
    ds.sizes["lon"],
)

# degrees
H = ds.coords["lat"].values[-1] - ds.coords["lat"].values[0]
# convert to meters
H = H * 111e3
dy = H / ds.sizes["lat"]

# lat axis
lat_axis = Axis(
    ds.coords["lat"].min().item(),
    ds.coords["lat"].diff("lat").mean().item(),
    ds.sizes["lat"],
)

print(f"{L = :.2e} m")
print(f"{dx = :.2e} m")
print(f"{H = :.2e} m")
print(f"{dy = :.2e} m")

# make a scaling object to convert between unit systems
scaling = Scaling(H)

dx = scaling.distance.convert(dx, UnitSystem.SI, UnitSystem.scaled)
dy = scaling.distance.convert(dy, UnitSystem.SI, UnitSystem.scaled)
print(f"{dx = :.2e} scaled")
print(f"{dy = :.2e} scaled")

# convert Fx and Fy to scaled units
Fx = scaling.water_vapor_flux.convert(Fx.values, UnitSystem.natural, UnitSystem.scaled)
Fy = scaling.water_vapor_flux.convert(Fy.values, UnitSystem.natural, UnitSystem.scaled)

# convert E to scaled units
# Do this for both the total E and the local regionally clipped E]
#print('pre-scaled',ds['Evap'])
E_total = scaling.evaporation.convert(ds["Evap_all"].values, UnitSystem.natural, UnitSystem.scaled)
E_local = scaling.evaporation.convert(ds["Evap_cb"].values, UnitSystem.natural, UnitSystem.scaled)

**Plot the scaled E array**


In [ ]:
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.RIVERS)
ax.add_feature(cartopy.feature.OCEAN)
#Overlay with all ward outline
shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
#Plot data
Ea = np.transpose(np.average(E_total,axis=2))
collection = plt.pcolormesh(ds.lon, ds.lat,Ea)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()

ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.RIVERS)
ax.add_feature(cartopy.feature.OCEAN)
#Overlay with all ward outline
shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
#Plot data
Ea = np.transpose(np.average(E_local,axis=2))
collection = plt.pcolormesh(ds.lon, ds.lat,Ea)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()

**Run recycling model for each timestep**
- Create recycling output array based on the shape of one of the surface input files: evap 
- Translate evap and fluxes to secondary grid
- Calculate modeled precipitation
- Plot scaled input variables (evap and fluxes)
- Run through each timestep in the input files and calculate recycling ratio at each timestep across domain
- Plot rho and convergence metric for each timestep

In [ ]:
import matplotlib.pyplot as plt
import logging
logging.basicConfig()
logging.getLogger("bulk_recycling_model").setLevel(logging.INFO)
from bulk_recycling_model import plotting
from bulk_recycling_model.main import run_4_orientations, run_rotated
from bulk_recycling_model import coefficients

max_iter = 1000
tol = 1e-3

#Make the rho array the same shape as the total E - will clip the external points at the end
rho_ar = np.empty((4,np.shape(E_total)[0]-1,np.shape(E_total)[1]-1,np.shape(E_total)[2]))
#Entering preprocessing and time step loop
#Run model and plot
count_success_pre_nudge = 0
count_success_post_nudge = 0
count_fail_pre_nudge = 0
count_fail_post_nudge = 0
too_many_pixels = 0
no_hot_pixel = 0
for i,time in enumerate(ds.time):
     
    # preprocess E onto the secondary grid
    Ei_total = preprocess.prepare_E(E_total[:,:,i])
    Ei_local = preprocess.prepare_E(E_local[:,:,i])
    
    #print('post-processed land min',Ei_local.min())
    #print('post-processed total min',Ei_total.min())
    
    # preprocess water vapor fluxes onto the secondary grid
    Fxi_left = preprocess.prepare_Fx_left(Fx[:,:,i])
    #print("***Fxi_left**")
    
    
    Fxi_right = preprocess.prepare_Fx_right(Fx[:,:,i])
    Fyi_bottom = preprocess.prepare_Fy_bottom(Fy[:,:,i])
    Fyi_top = preprocess.prepare_Fy_top(Fy[:,:,i])
    
    # compute P
    Pi = preprocess.calculate_precipitation(Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, Ei_total, dx, dy)

    # Create a quiver plot
    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
    ax.coastlines()
    ax.add_feature(cartopy.feature.BORDERS)
    ax.add_feature(cartopy.feature.RIVERS)
    ax.add_feature(cartopy.feature.OCEAN)
    collection = plotting.pcolormesh(ax, Ei_local, lon_axis, lat_axis, alpha=0.5)
    fig.colorbar(collection, label="E (scaled)")
    plotting.quiver(ax, Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, lon_axis, lat_axis)
    fig.suptitle("Evaporation + Water Vapor Fluxes on cell edges")
    #plt.show()
    plt.clf()
    
    # Create a precip plot
    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
    ax.coastlines()
    ax.add_feature(cartopy.feature.BORDERS)
    ax.add_feature(cartopy.feature.RIVERS)
    ax.add_feature(cartopy.feature.OCEAN)
    cmap=plt.cm.viridis
    cmap.set_extremes(over='yellow')
    cmap.set_extremes(under='red')
    
    collection = plotting.pcolormesh(ax, Pi, lon_axis, lat_axis, 
                                     alpha=1,cmap=cmap,
                                     vmin=0.0,vmax=20.0)
    fig.colorbar(collection, label="P (calculated)",extend='min')
    shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
    #plotting.quiver(ax, Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, lon_axis, lat_axis)
    fig.suptitle("**BASELINE** Precipitation")
    plt.show()
    plt.clf()
    
    # Run the model
    status = run_4_orientations(
        Fxi_left,
        Fxi_right,
        Fyi_bottom,
        Fyi_top,
        Ei_local,
        Pi,
        dx,
        dy,
        R=0.2,
        R_1=0.2,
        max_iter=max_iter,
        tol=tol,
    )
    #Print timestep and status (converged or not) and add rho to recycling ration array
    print(i,time.values)
    for ROT in np.arange(0,4):
        print(ROT,status[ROT]['success'],'**pre-nudge')    
        if status[ROT]['success']==True:
            rho_ar[ROT,:,:,i] = status[ROT]["rho"]
            count_success_pre_nudge = count_success_pre_nudge + 1
        else:
            rho_ar[ROT,:,:,i] = np.nan
            count_fail_pre_nudge = count_fail_pre_nudge + 1
            
            print("***PRE-NUDGE PRINT***")
            print(np.argwhere(ds['lat'].values==7.0))
            print(np.argwhere(ds['lon'].values==27.0))
            print('Ei = ',Ei_local[28,8])
            print('Pi = ',Pi[28,8])
            print('Fxi left = ',Fxi_left[28,8])
            print('Fxi right= ',Fxi_right[28,8])
            print('Fyi top= ',Fyi_top[28,8])
            print('Fyi bottom= ',Fyi_bottom[28,8])
            print('dx= ',dx)
            print('dy= ',dy)
            
            
    
            
            #----Pre-nudge------------------------
            
            coeffs = coefficients.Coefficients(Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, Ei_local, Pi, dx,dy)
            instability_heuristic = coeffs.rotated_instability_heuristic(k=ROT)

            # plot the convergence
            deltas = status[ROT]["deltas"]
            fig, ax = plt.subplots()
            ax.plot(deltas)
            ax.set_title("Rot: "+str(ROT)+" - Convergence")
            ax.set_xlabel("Iteration")
            plt.show()
            plt.close()
            
            #identify hot pixel
            from bulk_recycling_model.numerical_stability_ED import identify_hot_pixel, identify_hot_pixel_thresh, nudge_hot_pixel
            #printing example of the hottest pixels
            p = 4
            s_p = 1
            thresh=1.75
            s_thresh = 1.2
            #tune these nudging values as needed
            offset = 2.0
            kernel_size = 15
            #hot_ind = identify_hot_pixel(p,coeffs.rotated_instability_heuristic(k=ROT))
            hot_ind = identify_hot_pixel_thresh(thresh,coeffs.rotated_instability_heuristic(k=ROT))
            if len(hot_ind)>p:
                hot_ind=hot_ind
            if len(hot_ind)==0:
                hot_ind = identify_hot_pixel_thresh(s_thresh,coeffs.rotated_instability_heuristic(k=ROT)) 
                if len(hot_ind)>0:
                    hot_ind = identify_hot_pixel(s_p,coeffs.rotated_instability_heuristic(k=ROT)) 
            print('*There are ',len(hot_ind),' hot pixels*')
            if len(hot_ind)>0 and len(hot_ind)<=p:
                i_hot, j_hot = np.unravel_index(np.argmax(instability_heuristic, axis=None), instability_heuristic.shape)
                print(
                    f"Hottest pixel identified at (i={i_hot}, j={j_hot}) "
                    f"with instability heuristic = {coeffs.rotated_instability_heuristic(k=ROT)[i_hot, j_hot]}"
                )
                print("E =", Ei_local[i_hot, j_hot])
                print("P =", Pi[i_hot, j_hot])
                
                # Create a evap plot
                fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                ax.coastlines()
                ax.add_feature(cartopy.feature.BORDERS)
                ax.add_feature(cartopy.feature.RIVERS)
                ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                collection = plotting.pcolormesh(ax, Ei_local, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=0.0,vmax=5.0)
                plt.scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Ei_local[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=0.0,vmax=5.0)
                fig.colorbar(collection,extend='both',location='bottom')
                fig.suptitle("E Local - pre-nudge")
                plt.show()
                plt.clf()
                
                #Flux subplot
                vmin = -10
                vmax = 10
                fig, axs = plt.subplots(2, 2)#, subplot_kw={'projection': ccrs.PlateCarree()})
                #ax.coastlines()
                #ax.add_feature(cartopy.feature.BORDERS)
                #ax.add_feature(cartopy.feature.RIVERS)
                #ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                p1 = plotting.pcolormesh(axs[0,0],Fxi_left, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=vmin,vmax=vmax)
                axs[0,0].set_title('Fxi_left')
                axs[0,0].scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Fxi_left[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=vmin,vmax=vmax)
                fig.colorbar(p1,extend='both',location='bottom')
                p2 = plotting.pcolormesh(axs[0,1], Fxi_right, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=vmin,vmax=vmax)
                axs[0,1].set_title('Fxi_right')
                axs[0,1].scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Fxi_right[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=vmin,vmax=vmax)
                fig.colorbar(p2,extend='both',location='bottom')
                p3 = plotting.pcolormesh(axs[1,0], Fyi_bottom, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=vmin,vmax=vmax)
                axs[1,0].set_title('Fyi_bottom')
                axs[1,0].scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Fyi_bottom[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=vmin,vmax=vmax)
                fig.colorbar(p3,extend='both',location='bottom')
                p4 = plotting.pcolormesh(axs[1,1], Fyi_top, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=vmin,vmax=vmax)
                axs[1,1].set_title('Fyi_top')
                axs[1,1].scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Fyi_top[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=vmin,vmax=vmax)
                fig.colorbar(p4,extend='both',location='bottom')
                fig.suptitle("Fluxes - pre-nudge")
                fig.tight_layout()
                plt.show()
                plt.clf()
    
                # Create a precip plot
                fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                ax.coastlines()
                ax.add_feature(cartopy.feature.BORDERS)
                ax.add_feature(cartopy.feature.RIVERS)
                ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                #cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                collection = plotting.pcolormesh(ax, Pi, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=-2.0,vmax=0.0)
                plt.scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=Pi[i_hot,j_hot],
                            edgecolors='red',cmap=cmap, vmin=-2.0,vmax=0.0)
                fig.colorbar(collection,extend='min',location='bottom')
                shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
                fig.suptitle("P calculated - pre-nudge")
                plt.show()
                plt.clf()
    
    
                fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                ax.coastlines()
                ax.add_feature(cartopy.feature.BORDERS)
                ax.add_feature(cartopy.feature.RIVERS)
                ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                collection = plotting.pcolormesh(ax, instability_heuristic, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap, vmin=1.2,vmax=2.0)
                plt.scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=coeffs.rotated_instability_heuristic(k=ROT)[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=1.2,vmax=2.0)
                fig.colorbar(collection,extend='both',location='bottom')
                shp_cod.plot(ax=ax, edgecolor='white',facecolor='none',lw=2,zorder=2,linestyle='-')
                fig.suptitle("Instability heuristic - pre-nudge")
                plt.show()
                plt.clf()
                plt.close()
    
                #nudge hot pixel
                #print('post-processed land min',Ei_local.min())
                #print('post-processed total min',Ei_total.min())
                E_local_nudged = nudge_hot_pixel(Ei_local, hot_ind, offset=offset, kernel_size=kernel_size)
                E_total_nudged = nudge_hot_pixel(Ei_total, hot_ind, offset=offset, kernel_size=kernel_size)
                P_nudged = preprocess.calculate_precipitation(Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, E_total_nudged, dx, dy)
                coeffs_nudged = coefficients.Coefficients(Fxi_left, Fxi_right, Fyi_bottom, Fyi_top, E_local_nudged, P_nudged, dx, dy)
                
                
                #----Post-nudge------------------------
                # Create a precip plot
                fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                ax.coastlines()
                ax.add_feature(cartopy.feature.BORDERS)
                ax.add_feature(cartopy.feature.RIVERS)
                ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                collection = plotting.pcolormesh(ax, P_nudged, lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap,
                                                 vmin=-2.0,vmax=0.0)
                plt.scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=P_nudged[i_hot,j_hot],
                            edgecolors='red',cmap=cmap, vmin=-2.0,vmax=0.0)
                fig.colorbar(collection,extend='both',location='bottom')
                shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
                fig.suptitle("P calculated - post-nudge")
                plt.show()
                plt.clf()
    
                fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                ax.coastlines()
                ax.add_feature(cartopy.feature.BORDERS)
                ax.add_feature(cartopy.feature.RIVERS)
                ax.add_feature(cartopy.feature.OCEAN)
                cmap=plt.cm.viridis
                cmap.set_extremes(over='orange')
                cmap.set_extremes(under='pink')
                collection = plotting.pcolormesh(ax, coeffs_nudged.rotated_instability_heuristic(k=ROT), lon_axis, lat_axis, 
                                                 alpha=1,cmap=cmap, vmin=1.2,vmax=2.0)
                plt.scatter(ds.coords["lon"][i_hot],ds.coords["lat"][j_hot],s=100,c=coeffs_nudged.rotated_instability_heuristic(k=ROT)[i_hot,j_hot],
                            edgecolors='white',cmap=cmap, vmin=1.2,vmax=2.0)
                fig.colorbar(collection,extend='both',location='bottom')
                shp_cod.plot(ax=ax, edgecolor='white',facecolor='none',lw=2,zorder=4,linestyle='-')
                fig.suptitle("Instability heuristic - post-nudge")
                plt.show()
                plt.clf()
                
                print(
                    f"Hottest pixel post-nudge at (i={i_hot}, j={j_hot}) "
                    f"with instability heuristic = {coeffs_nudged.rotated_instability_heuristic(k=ROT)[i_hot, j_hot]}"
                    )
                print("E =", E_local_nudged[i_hot, j_hot])
                print("P =", P_nudged[i_hot, j_hot])
    
                # Run the model again
                status_or = run_rotated(
                    Fxi_left,
                    Fxi_right,
                    Fyi_bottom,
                    Fyi_top,
                    E_local_nudged,
                    P_nudged,
                    dx,
                    dy,
                    R=0.2,
                    R_1=0.2,
                    max_iter=max_iter,
                    tol=tol,
                    rotation=ROT,
                )
                #Print timestep and status (converged or not) and add rho to recycling ration array
                print(i,time.values)
                print(ROT,status_or['success'],'**post-nudge')    
                if status_or['success']==True:
                    rho_ar[ROT,:,:,i] = status_or["rho"]
                    count_success_post_nudge = count_success_post_nudge + 1
                else:
                    rho_ar[ROT,:,:,i] = np.nan 
                    count_fail_post_nudge = count_fail_post_nudge + 1
                
                    lon_ar = np.linspace(start=ds.coords["lon"].min().values+lon_axis.step/2,
                             stop=ds.coords["lon"].max().values-lon_axis.step/2,
                             num=lon_axis.n_points-1)
                    lat_ar = np.linspace(start=ds.coords["lat"].min().values+lat_axis.step/2,
                                         stop=ds.coords["lat"].max().values-lat_axis.step/2,
                                         num=lat_axis.n_points-1)
                    rho_xarr = xr.Dataset(
                        data_vars=dict(rho=(["lon","lat"],status_or["rho"])),
                        coords=dict(
                            lon=(["lon"], lon_ar),
                            lat=(["lat"], lat_ar)
                        ),
                        attrs=dict(
                            description="Recycling ratio",
                            units="%",
                        ),
                    ) 
                    rho_xarr = rho_xarr.transpose("lat","lon")
                    
#                    if B in ['EQ','S']:
#                        mask_surf = mask.interp(lat=rho_xarr['lat'],lon=rho_xarr['lon'],method='linear',kwargs={"fill_value": "extrapolate"})
#                        rho_xarr = rho_xarr.where(mask_surf!=0.0,0,np.nan)
            
                    fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
                    ax.coastlines()
                    ax.add_feature(cartopy.feature.BORDERS)
                    ax.add_feature(cartopy.feature.RIVERS)
                    #ax.add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')
                    cmap=plt.cm.viridis
                    cmap.set_extremes(over='orange')
                    cmap.set_extremes(under='pink')
                    collection = rho_xarr['rho'].plot.pcolormesh(vmin=0.0,vmax=1.0,
                                                                    #levels=13,
                                                                    ax=ax,extend='both',cmap=cmap,
                                                                    cbar_kwargs={"location":"bottom"} )
                    shp_cod.plot(ax=ax, edgecolor='pink',facecolor='none',lw=2,zorder=2,linestyle='-')
                    ax.set_extent([8, 31, -8, 8])
                    ax.set_title(" $\\rho$"+str(time.values)+' '+str(YR)+"\n CB (rot "+str(ROT)+")")
                    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=1, color='black', alpha=0.5, linestyle='dotted',zorder=100)
                    gl.top_labels = False
                    gl.right_labels = False
                    plt.show()
                    plt.clf()
                    plt.close()
            else:
                print('No hot pixels above threshold or there are too many hot pixels')
                if len(hot_ind)==0: 
                  no_hot_pixel = no_hot_pixel + 1 
                if len(hot_ind)>p:
                  too_many_pixels = too_many_pixels + 1  
                    
print('count_success_pre_nudge: ', count_success_pre_nudge)
print('count_fail_pre_nudge: ', count_fail_pre_nudge)
print('count_success_post_nudge: ', count_success_post_nudge)
print('count_fail_post_nudge: ', count_fail_post_nudge)
print('count_fail_no_hot_pixel: ', no_hot_pixel)
print('count_fail_too_many_pixels: ', too_many_pixels)

**Create and save rho xarray file**

- Create an xarray to store all of the calculated recycling ratios that is organised in an easy to plot/interpret format
- Count number of values in array over 1 - replace all of these with 1
- Count number of negative rho values - replace all of these with zero
- Save to file

In [ ]:
lon_ar = np.linspace(start=ds.coords["lon"].min().values+lon_axis.step/2,
                     stop=ds.coords["lon"].max().values-lon_axis.step/2,
                     num=lon_axis.n_points-1)
lat_ar = np.linspace(start=ds.coords["lat"].min().values+lat_axis.step/2,
                     stop=ds.coords["lat"].max().values-lat_axis.step/2,
                     num=lat_axis.n_points-1)
rho_xarr = xr.Dataset(
    data_vars=dict(rho=(["rot","lon","lat","time"],rho_ar)),
    coords=dict(
        rot=(["rot"],[1,2,3,4]),
        lon=(["lon"], lon_ar),
        lat=(["lat"], lat_ar),
        time=(["time"],ds.time.data)
    ),
    attrs=dict(
        description="Recycling ratio",
        units="%",
    ),
) 
rho_xarr = rho_xarr.transpose("rot","time","lat","lon")
rho_xarr = rho_xarr.rio.set_spatial_dims(x_dim="lon",y_dim="lat")
rho_xarr.rio.write_crs("epsg:4326", inplace=True)
rho_xarr = rho_xarr.rio.clip(shp_cod.geometry.apply(mapping),shp_cod.crs,drop=False)
rho_xarr.to_netcdf(datao+L_NAME+"_"+S_NAME+"_cb_rot_rho_era5_"+str(YR)+".nc")

In [ ]:
##Filtering out outliers for plotting 
#for r in np.arange(1,5):
#    print("*** Rotation is: ",r)
#    print('Number of rhos over 1: ', rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values>1.0).count().values)
#    print('Number of negative rhos: ', rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values<0.0).count().values)
#    rho_xarr['rho'][r-1,:,:,:] = rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values<=1.0,1.0)
#    rho_xarr['rho'][r-1,:,:,:] = rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values>0.0,0.0)
#    print('Number of rhos over 1: ', rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values>1.0).count().values)
#    print('Number of negative rhos: ', rho_xarr['rho'][r-1,:,:,:].where(rho_xarr['rho'][r-1,:,:,:].values<0.0).count().values)
#    print('-------------------------')
#end_all = timer.time()
#length = end_all - start_all
#print("Running the whole prep and recycling code took ", length, "seconds")
#

**Plotting**

Create seasonal arrays and plot these

In [ ]:
# Create seasonal arrays and plot these
seas={'MAM':[3,4,5],'SON':[9,10,11],'JJA':[6,7,8],'D':[12],'JF':[1,2]}
for S in seas:
    for r in np.arange(1,5):
        rho_plot = rho_xarr['rho'][r-1,:,:,:] 
        seas_rho = rho_plot.sel(time=rho_xarr.time.dt.month.isin(seas[S]))
        fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
        ax.coastlines()
        ax.add_feature(cartopy.feature.BORDERS)
        ax.add_feature(cartopy.feature.RIVERS)
        ax.add_feature(cartopy.feature.OCEAN,zorder=2,facecolor='lightgrey')
        cmap=plt.cm.viridis
        cmap.set_extremes(over='white')
        collection = seas_rho.mean("time").plot.pcolormesh(vmin=0.0,vmax=1.0,
                                                        #levels=13,
                                                        ax=ax,extend='max',cmap=cmap,
                                                        cbar_kwargs={"location":"bottom"} )
        ax.set_extent([7, 31, -8, 8])
        ax.set_title(" $\\rho$"+" "+S+" "+str(YR)+"\n CB (rot "+str(r)+")")
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
              linewidth=1, color='black', alpha=0.5, linestyle='dotted',zorder=100)
        gl.top_labels = False
        gl.right_labels = False
        plt.savefig(datap+L_NAME+"_"+S_NAME+"_rho_"+S+"_"+str(YR)+"_CB_rot"+str(r)+".png")
        plt.show()
        
rho_xarr.close()
